In [ ]:
%pip install


## 0. Import Libraries and Services

In [ ]:
from dataclasses import dataclass
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    default_data_collator,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    PreTrainedModel,
)
import os
from pathlib import Path
import zipfile
from torch.utils.data import Dataset
from typing import Dict, List
from matplotlib import pyplot as plt
import pandas as pd
from pandas import DataFrame
from PIL import Image as PILImage
from datasets import Dataset as HFDataset, Image
import torch
import numpy as np
from datasets import load_from_disk, tqdm, DatasetDict
import random
import evaluate


## 1. Configuration

In [ ]:
@dataclass
class OCRConfig:
    BASE_MODEL: str = "microsoft/trocr-large-handwritten"
    PROJECT_ROOT: str = "/content/drive/MyDrive/medicare_plus_ocr"
    DATA_ROOT: str = f"{PROJECT_ROOT}/data"

    MAX_LENGTH: int = 128
    NUM_BEAMS: int = 5

    BATCH_SIZE: int = 16
    LEARNING_RATE = 2e-5
    EPOCHS: int = 30

    FP16: bool = True

    TRAINING_DIR: str = f"{DATA_ROOT}/raw/Training"
    TESTING_DIR: str = f"{DATA_ROOT}/raw/Testing"
    VALIDATION_DIR: str = f"{DATA_ROOT}/raw/Validation"

    MODEL_DIR: str = f"{PROJECT_ROOT}/models"
    LOG_DIR: str = f"{PROJECT_ROOT}/logs"

    TRAINING_ARGS = Seq2SeqTrainingArguments(
        output_dir=f"{LOG_DIR}/checkpoints",
        overwrite_output_dir=True,
        predict_with_generate=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        # GPU Power E.g. 4GB
        per_device_train_batch_size=12,
        per_device_eval_batch_size=8,
        fp16=torch.cuda.is_available(),
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        warmup_steps=200,
        weight_decay=0.01,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="cer",
        greater_is_better=False,
        report_to="all",
    )


## 2. Preprocessing Phase

### 2.1 Data Loader

In [ ]:
class Dataset_Loader:
    def __init__(self) -> None:
        self.required_dirs = ["Training_Data", "Testing_Data", "Validation_Data"]

    def verify_root(self, root: str | Path) -> str:
        try:
            if not os.path.isfile(root):
                return "File doesn't exists!"
            os.makedirs(root, exist_ok=True)
            return "File created in root!"
        except Exception as e:
            return str(RuntimeError(f"Failed to set up Google Drive: {e}"))

    def extract_file(
        self,
        file_path: str | Path,
        extract_path: str | Path,
    ) -> str:
        try:
            if not os.path.isfile(file_path):
                return f"Error: File_Path must be a not found"
            if not str(file_path).endswith(".zip"):
                return f".zip file not found!"

            with zipfile.ZipFile(file_path, "r") as file:
                file.extractall(extract_path)
            for dir_name in self.required_dirs:
                path = os.path.join(extract_path, dir_name)
                if not os.path.isdir(path):
                    return f"Required directory not found after extraction: {path}"

            return "Dataset verified and extracted successfully."
        except zipfile.BadZipFile:
            return "Error: The file is not a valid zip file."
        except Exception as e:
            return f"An unexpected error occurred during extraction: {e}"

    def verify_csv(
        self,
        file_dir: str | Path,
        extensions: set = {".csv"},
    ) -> List[Path]:
        raw = Path(file_dir)
        if not raw.exists():
            raise FileNotFoundError(f"Directory '{file_dir}' does not exist.")
        files = []

        for ext in extensions:
            files.extend(raw.rglob(f"*{ext}"))

        csv_files = [file for file in files]
        print(f"CSV Files > {len(csv_files)}")
        return csv_files

    def verify_img(
        self,
        file_dir: str | Path,
        extensions: set = {".png", ".jpg", ".jpeg"},
    ) -> List[Path]:
        raw = Path(file_dir)
        if not raw.exists():
            raise FileNotFoundError(f"Directory '{file_dir}' Does Not Exist.")

        image_files = [p for p in raw.rglob("*") if p.suffix.lower() in extensions]
        print(f"Image Files > {len(image_files)}")
        return image_files


### 2.2 Return Special Types in Datasets

In [ ]:
class OCRDataset(Dataset):
    def __init__(self, hf_split, processor, max_target_length=64):
        self.split = hf_split
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        row = self.split[idx]
        image = row["image"].convert("RGB")
        text = row["label"]
        pixel_values = self.processor(image, return_tensors="pt").pixel_values[0]
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids
        labels = [
            t if t != self.processor.tokenizer.pad_token_id else -100 for t in labels
        ]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


### 2.3 Process of Pre-Processing

In [ ]:
class Preprocessing:
    def __init__(self):
        pass

    def _normalize_label(self, s: str) -> str:
        return str(s).strip().lower()

    def _lower_case(self, df: DataFrame) -> Dict[str, str]:
        return {c.lower(): c for c in df.columns}

    def _image_column_reader(self, column: Dict) -> str | None:
        return next(
            (column[k] for k in column if "image" in k or "file" in k or "name" in k),
            None,
        )

    def _label_column_reader(self, column: Dict) -> str | None:
        return next(
            (
                column[k]
                for k in column
                if "medicine" in k or "label" in k or "word" in k or "text" in k
            ),
            None,
        )

    def load_files(
        self,
        csv_path: List,
        image_path: List,
    ) -> DataFrame:
        records: List = []
        for csv_file in csv_path:
            try:
                df = pd.read_csv(csv_file)
                cols_lower = self._lower_case(df)
                img_col = self._image_column_reader(cols_lower)
                label_col = self._label_column_reader(cols_lower)
                if img_col and label_col and img_col != label_col:
                    print(f"""
                        File Records:\n 
                        \tPath: {csv_path}\n  
                        \tImage Column Count: {img_col}\n  
                        \tLabel Column Count: {label_col}
                    """)
                    path_lookup = {p.name: p for p in image_path}

                    for _, row in df.iterrows():
                        img_name = str(row[img_col]).strip()
                        label = self._normalize_label(row[label_col])
                        p = (
                            path_lookup.get(img_name)
                            or path_lookup.get(img_name + ".png")
                            or path_lookup.get(img_name + ".jpg")
                        )
                        if p is not None and label:
                            records.append({"image_path": str(p), "label": label})
                    break
            except Exception as e:
                print(f"Error: {e}")
        return (
            DataFrame(records)
            .drop_duplicates(subset=["image_path"])
            .reset_index(drop=True)
        )

    def display_dataset_info(self, df: DataFrame):
        print(f"Total samples: {len(df)}")
        print(f'Unique labels: {df["label"].nunique()}')
        df.head()

    def display_dataset_explore(self, df: DataFrame):
        label_counts = df["label"].value_counts()
        print(f"Drug Names: {label_counts.__len__()}")
        sample = df.sample(min(12, len(df)), random_state=42).reset_index(drop=True)
        fig, axes = plt.subplots(3, 4, figsize=(14, 8))
        for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
            img = PILImage.open(row["image_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(row["label"], fontsize=10)
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    def to_hf(self, df) -> HFDataset:
        ds = HFDataset.from_pandas(
            df[["image_path", "label"]].rename(columns={"image_path": "image"})
        )
        return ds.cast_column("image", Image())


## 3. Training Phase

### 3.1 Setup Model for Training Process

In [ ]:
class TrOCRModel:

    def __init__(
        self,
        processor: TrOCRProcessor,
        model: VisionEncoderDecoderModel | PreTrainedModel,
        config: OCRConfig,
    ):
        self.processor = processor
        self.model = model
        self.cer_metric = evaluate.load("cer")

        self._check_resources()

        if os.makedirs(
            config.MODEL_DIR,
            exist_ok=True,
        ) and os.makedirs(
            config.LOG_DIR,
            exist_ok=True,
        ):
            return None

    def _check_resources(self) -> None:
        print(f"PyTorch: {torch.__version__}")
        print(f"CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"Device: {torch.cuda.get_device_name(0)}")

    def setup_model(self, config: OCRConfig):
        tok = self.processor.tokenizer  # type: ignore
        self.model.config.decoder_start_token_id = tok.bos_token_id or tok.cls_token_id
        self.model.config.pad_token_id = tok.pad_token_id
        self.model.config.eos_token_id = tok.eos_token_id or tok.sep_token_id

        self.model.config.max_length = config.MAX_LENGTH
        self.model.config.num_beams = config.NUM_BEAMS


### 3.2 Process of Model Training

In [ ]:
class OCRTrainer:
    def __init__(self, model_obj, config: OCRConfig, training_data, validation_data):
        self.model_obj = model_obj
        self.config = config
        self.cer_metric = evaluate.load("cer")

        self.seq_model = Seq2SeqTrainer(
            model=self.model_obj,
            args=config.TRAINING_ARGS,
            train_dataset=training_data,
            eval_dataset=validation_data,
            processing_class=self.model_obj.processor.feature_extractor,
            compute_metrics=self._compute_metrics,
            data_collator=default_data_collator,
        )

    def _compute_metrics(self, eval_pred) -> Dict:
        pred_ids, label_ids = eval_pred
        label_ids = np.where(
            label_ids != -100,
            label_ids,
            self.processor.tokenizer.pad_token_id,  # type: ignore
        )
        self.label_str = self.model_obj.processor.batch_decode(
            label_ids,
            skip_special_tokens=True,
        )
        self.pred_str = self.model_obj.processor.batch_decode(
            pred_ids,
            skip_special_tokens=True,
            predict_with_generate=True,
        )
        cer = self.cer_metric.compute(
            predictions=self.pred_str,
            references=self.label_str,
        )
        return {"cer": cer}

    def get_preds(self):
        return self.pred_str

    def get_refs(self):
        return self.label_str

    def train(self) -> Seq2SeqTrainer:
        return self.seq_model.train()

    def save_model(self):
        self.seq_model.save_model(self.config.MODEL_DIR)
        self.model_obj.processor.save_pretrained(self.config.MODEL_DIR)
        print(f"Model saved to: {self.config.MODEL_DIR}")


## 4. Model Evaluation Phase

In [ ]:
class Predictor:
    def __init__(self, model, processor, config: OCRConfig):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.config = config
        self.processor = processor
        self.model = model.to(self.device)

    def predict(self, image):
        pixel_values = self.processor(
            image,
            return_tensors="pt",
        ).pixel_values.to(self.device)
        with torch.no_grad():
            ids = self.model.generate(
                pixel_values,
                max_length=self.config.MAX_LENGTH,
                num_beams=self.config.NUM_BEAMS,
                num_return_sequences=1,
                early_stopping=True,
            )
        return self.processor.batch_decode(
            ids,
            skip_special_tokens=True,
        )[0]


### 4.1 Pre Evaluation for Make Predictions and Outcomes

In [ ]:
class PreEvaluation:
    def __init__(self, config: OCRConfig, predictor: Predictor) -> None:
        self.config = config
        self.predictor = predictor

    def verify_dataset(self):
        splits = load_from_disk(self.config.TESTING_DIR)
        print(f"Testign Samples: {len(splits)}")
        return splits

    def load_dataset(self, dataset, preds, refs, batch_size=16):
        for i in tqdm(range(0, len(dataset), batch_size)):
            rows = dataset[i : i + batch_size]
            images = [img.convert("RGB") for img in rows["image"]]
            labels = [str(l).strip().lower() for l in rows["label"]]
            out = self.predictor.predict(images)
            preds.extend([t.strip().lower() for t in out])
            refs.extend(labels)
        results_df = DataFrame({"reference": refs, "prediction": preds})
        return results_df


### 4.2 Post Evaluation for Generate the Matrices and Values

In [ ]:
class PostEvaluation:
    def __init__(self) -> None:
        self.cer_metric = evaluate.load("cer")
        self.wer_metric = evaluate.load("wer")

    def compute_matrices(self, preds, refs) -> str:
        cer = self.cer_metric.compute(predictions=preds, references=refs)
        wer = self.wer_metric.compute(predictions=preds, references=refs)
        return f"Test CER: {cer:.4f} | Test WER: {wer:.4f}"
        # return {"cer": cer, "wer": wer}

    def generate_accuracy(self, results_df: DataFrame) -> str:
        exact = (results_df["reference"] == results_df["prediction"]).mean()
        return f"Accuracy: {exact:.5f} -> ({int(exact * len(results_df))} / {len(results_df)})"

    def get_common_errors(self, results_df: DataFrame):
        errors_df = results_df[
            results_df["reference"] != results_df["prediction"]
        ].copy()
        errors_df["pair"] = errors_df["reference"] + "  →  " + errors_df["prediction"]
        return f"""
            Total errors: {len(errors_df)} / {len(results_df)}
            Top 20 Most Frequent Confusion Pairs:\n
            {errors_df['pair'].value_counts().head(20).to_string()}
        """

    def matrix_visualization(self, dataset, preds, n_show=12):
        sample_idx = random.sample(range(len(dataset)), n_show)
        fig, axes = plt.subplots(3, 4, figsize=(14, 9))
        for ax, idx in zip(axes.ravel(), sample_idx):
            row = dataset[idx]
            img = row["image"].convert("RGB")
            label = str(row["label"]).strip().lower()
            pred = preds[idx]
            color = "green" if pred == label else "red"
            ax.imshow(img)
            ax.set_title(f"true: {label}\npred: {pred}", fontsize=9, color=color)
            ax.axis("off")
        plt.tight_layout()
        plt.show()


In [ ]:
if __name__ == "__main__":

    __title__ = "TrOCR"
    __version__ = "1.0.0"

    config = OCRConfig()

    dl = Dataset_Loader()
    file_exract = dl.extract_file(config.PROJECT_ROOT, config.PROJECT_ROOT)

    train_csv_files = dl.verify_csv(config.TRAINING_DIR)
    train_img_files = dl.verify_img(config.TRAINING_DIR)

    val_csv_files = dl.verify_csv(config.TRAINING_DIR)
    val_img_files = dl.verify_img(config.TRAINING_DIR)

    dp = Preprocessing()

    train_df = dp.load_files(train_csv_files, train_img_files)
    train_df_info = dp.display_dataset_info(train_df)
    print(train_df_info)
    train_df_explore = dp.display_dataset_explore(train_df)
    print(train_df_explore)
    train_convert_hf = dp.to_hf(train_df)

    val_df = dp.load_files(val_csv_files, val_img_files)
    val_df_info = dp.display_dataset_info(train_df)
    print(val_df_info)
    val_df_explore = dp.display_dataset_explore(val_df)
    print(val_df_explore)
    val_convert_hf = dp.to_hf(val_df)

    processor = TrOCRProcessor.from_pretrained(config.BASE_MODEL, use_fast=True)
    model = VisionEncoderDecoderModel.from_pretrained(
        config.BASE_MODEL,
        use_fast=True,
        use_auth_token=True,
    )

    tune_model = TrOCRModel(processor, model, config)
    config_tune_model = tune_model.setup_model(config)

    train_data = OCRDataset(train_convert_hf, processor)
    validation_data = OCRDataset(val_convert_hf, processor)

    trainer = OCRTrainer(model, config, train_data, validation_data)
    preds = trainer.get_preds()
    refs = trainer.get_refs()
    trained_model = trainer.train()
    trainer.save_model()

    predictor = Predictor(trained_model, processor, config)

    evaluate = PreEvaluation(config, predictor)
    test_data = evaluate.verify_dataset()
    result_df = evaluate.load_dataset(test_data, preds, refs)

    matrix = PostEvaluation()
    cm_matrix = matrix.compute_matrices(preds, refs)
    print(cm_matrix)
    gen_acc = matrix.generate_accuracy(result_df)
    print(gen_acc)
    get_common_err = matrix.get_common_errors(result_df)
    print(get_common_err)
    visualize_evaluation = matrix.matrix_visualization(test_data, preds)


## 6. Generate Output

In [ ]:
from typing import Any
from PIL import Image

params = OCRConfig()


def gen_text(file: Any, model: Any, preprocessor: Any):
    image = Image.open(file.file).convert("RGB")
    predictor = Predictor(model, preprocessor, params)
    output = predictor.predict(image=image)
    return {"text": output}


print(gen_text)
